In [1]:
# gnn_patient_classifier.py
from __future__ import annotations
import os, math, numpy as np
from typing import Optional, Tuple, List

import torch
import torch.nn.functional as F
from torch import nn

from torch_geometric.data import Dataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (
    GCNConv,
    TransformerConv,
    global_mean_pool,
    BatchNorm,
)

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm


/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------------------
# Dataset that reads .pt files (CPU only)
# -------------------------
class PatientGraphDataset(Dataset):
    """
    Each file should contain a PyG Data with:
      - x:          (n_nodes, feat_dim)
      - edge_index: (2, E)
      - edge_weight: (E,)   [for GCN]  (optional)
      - edge_attr:   (E,d)  [for Transformer] (optional)
      - y: graph label 0/1 (scalar or tensor shape [1])
    """
    def __init__(self, root: str, transform=None, pre_transform=None, map_location: str = "cpu"):
        super().__init__(root=root, transform=transform, pre_transform=pre_transform)
        self.root_dir = root
        self.files = sorted([os.path.join(root, f) for f in os.listdir(root) if f.endswith(".pt")])
        self.map_location = map_location
        self._labels_cache: Optional[List[int]] = None

    def len(self) -> int:
        return len(self.files)

    def get(self, idx: int) -> Data:
        obj = torch.load(self.files[idx], weights_only=False, map_location=self.map_location)
        data = obj[0] if isinstance(obj, (tuple, list)) else obj
        # ensure y is tensor [1]
        if not isinstance(data.y, torch.Tensor):
            data.y = torch.tensor([int(data.y)], dtype=torch.long)
        elif data.y.ndim == 0:
            data.y = data.y.unsqueeze(0)
        # ensure types
        if hasattr(data, "edge_weight") and data.edge_weight is not None:
            if data.edge_weight.ndim > 1:
                data.edge_weight = data.edge_weight.view(-1)
        if hasattr(data, "edge_attr") and data.edge_attr is not None and data.edge_attr.ndim == 1:
            data.edge_attr = data.edge_attr.view(-1, 1)
        return data

    def labels(self) -> List[int]:
        # cache to avoid re-loading later
        if self._labels_cache is not None:
            return self._labels_cache
        ys: List[int] = []
        for path in self.files:
            obj = torch.load(path, weights_only=False, map_location="cpu")
            data = obj[0] if isinstance(obj, (tuple, list)) else obj
            y = int(data.y.item()) if isinstance(data.y, torch.Tensor) else int(data.y)
            ys.append(y)
            # free ASAP
            del obj, data
        self._labels_cache = ys
        return ys

In [3]:
# --------------------------------
# Edge helpers
# --------------------------------
def get_edge_inputs_for_gcn(data: Data) -> Tuple[torch.Tensor, torch.Tensor]:
    ei = data.edge_index
    if hasattr(data, "edge_weight") and data.edge_weight is not None:
        ew = data.edge_weight
        return ei, ew if ew.ndim == 1 else ew.view(-1)
    if hasattr(data, "edge_attr") and data.edge_attr is not None and data.edge_attr.size(1) == 1:
        return ei, data.edge_attr.view(-1)
    return ei, torch.ones(ei.size(1), dtype=data.x.dtype, device=data.x.device)


def get_edge_inputs_for_transformer(data: Data) -> Tuple[torch.Tensor, torch.Tensor]:
    ei = data.edge_index
    if hasattr(data, "edge_attr") and data.edge_attr is not None:
        ea = data.edge_attr
        return ei, ea if ea.ndim == 2 else ea.view(-1, 1)
    if hasattr(data, "edge_weight") and data.edge_weight is not None:
        return ei, data.edge_weight.view(-1, 1)
    # fallback: unit edge feature
    return ei, torch.ones(ei.size(1), 1, dtype=data.x.dtype, device=data.x.device)


In [4]:
# -------------------------
# Models
# -------------------------
class GCN_Classifier(nn.Module):
    def __init__(self, in_ch=40, hidden=64, num_classes=2, dropout=0.2):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hidden, normalize=True)
        self.conv2 = GCNConv(hidden, hidden, normalize=True)
        self.bn1 = BatchNorm(hidden)
        self.bn2 = BatchNorm(hidden)
        self.head = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, data: Data):
        x, batch = data.x, data.batch
        edge_index, edge_weight = get_edge_inputs_for_gcn(data)

        x = self.conv1(x, edge_index, edge_weight).relu()
        x = self.bn1(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index, edge_weight).relu()
        x = self.bn2(x)

        x = global_mean_pool(x, batch)
        return self.head(x)

class Transf_Classifier(nn.Module):
    def __init__(self, in_ch=40, hidden=64, num_classes=2, dropout=0.2, edge_dim: Optional[int] = None):
        super().__init__()
        self.edge_dim = edge_dim
        self.conv1: Optional[TransformerConv] = None
        self.conv2: Optional[TransformerConv] = None
        self.bn1 = BatchNorm(hidden)
        self.bn2 = BatchNorm(hidden)
        self.head = nn.Linear(hidden, num_classes)
        self.dropout = dropout
        self.in_ch = in_ch
        self.hidden = hidden

    def _ensure_convs(self, edge_dim: int, device: torch.device):
        # (Re)build convs if missing or edge_dim changed; then move to device.
        if (self.conv1 is None) or (self.edge_dim != edge_dim):
            self.edge_dim = edge_dim
            self.conv1 = TransformerConv(self.in_ch, self.hidden, heads=1, edge_dim=edge_dim)
            self.conv2 = TransformerConv(self.hidden, self.hidden, heads=1, edge_dim=edge_dim)
            # IMPORTANT: move newly created modules to current device
            self.conv1.to(device)
            self.conv2.to(device)

    def forward(self, data: Data):
        x, batch = data.x, data.batch
        edge_index, edge_attr = get_edge_inputs_for_transformer(data)   # ensures 2D edge_attr
        self._ensure_convs(edge_attr.size(1), x.device)

        x = self.conv1(x, edge_index, edge_attr).relu()
        x = self.bn1(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index, edge_attr).relu()
        x = self.bn2(x)

        x = global_mean_pool(x, batch)
        return self.head(x)


In [5]:
# -------------------------
# Train / eval
# -------------------------
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    ys, preds, logits_all = [], [], []
    for data in loader:
        data = data.to(device)
        out = model(data)
        preds.append(out.argmax(dim=-1).cpu().numpy())
        ys.append(data.y.view(-1).cpu().numpy())
        logits_all.append(out.detach().cpu())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    logits = torch.vstack(logits_all)
    prob = logits.softmax(dim=-1)[:, 1].numpy() if logits.size(1) > 1 else logits.squeeze(1).sigmoid().numpy()

    acc = accuracy_score(y_true, y_pred)
    if len(np.unique(y_true)) == 2:
        auroc = roc_auc_score(y_true, prob)
    else:
        auroc = float("nan")
    return {"acc": acc, "auroc": auroc}


def train_one_epoch(model, loader, optimizer, device, class_weights: Optional[torch.Tensor] = None):
    model.train()
    crit = nn.CrossEntropyLoss(weight=class_weights)
    total = 0.0
    for data in loader:
        data = data.to(device)
        out = model(data)
        loss = crit(out, data.y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += float(loss.detach().cpu())
    return total / max(1, len(loader))


In [6]:
# -------------------------
# Main
# -------------------------
def main(
    root_dir: str,
    model_name: str = "gcn",    # "gcn" or "transformer",
    hidden: int = 64,
    dropout: float = 0.2,
    batch_size: int = 16,
    max_epochs: int = 100,
    lr: float = 1e-3,
    weight_decay: float = 5e-4,
    val_ratio: float = 0.2,
    test_ratio: float = 0.2,
    seed: int = 42,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    dataset = PatientGraphDataset(root_dir, map_location="cpu")
    if dataset.len() == 0:
        raise RuntimeError(f"No .pt files found in {root_dir}")

    labels = np.array(dataset.labels())   # now loaded on CPU
    idx_all = np.arange(len(labels))

    # split by files (patients)
    idx_train, idx_tmp, y_train, y_tmp = train_test_split(
        idx_all, labels, test_size=val_ratio + test_ratio, stratify=labels, random_state=seed
    )
    rel_test = test_ratio / (val_ratio + test_ratio) if (val_ratio + test_ratio) > 0 else 0.0
    idx_val, idx_test, y_val, y_test = train_test_split(
        idx_tmp, y_tmp, test_size=rel_test,
        stratify=y_tmp if len(np.unique(y_tmp)) > 1 else None, random_state=seed,
    )

    train_set = torch.utils.data.Subset(dataset, idx_train.tolist())
    val_set   = torch.utils.data.Subset(dataset, idx_val.tolist())
    test_set  = torch.utils.data.Subset(dataset, idx_test.tolist())

    # num_workers=0 keeps memory simple; increase if you want speed & have RAM
    pin_mem = (device.type == "cuda")
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,  pin_memory=pin_mem, num_workers=0)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False, pin_memory=pin_mem, num_workers=0)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False, pin_memory=pin_mem, num_workers=0)

    # class weights (optional for imbalance)
    classes, counts = np.unique(y_train, return_counts=True)
    cw = None
    if len(classes) == 2 and (counts.min() / counts.max()) < 0.8:
        w = torch.tensor([counts.max()/counts[0], counts.max()/counts[1]], dtype=torch.float32, device=device)
        cw = w
        print(f"Using class weights: {w.tolist()}")

    # model
    if model_name.lower() == "gcn":
        model = GCN_Classifier(in_ch=40, hidden=hidden, num_classes=2, dropout=dropout).to(device)
    elif model_name.lower() in ("transformer", "transformerconv", "transf"):
        model = Transf_Classifier(in_ch=40, hidden=hidden, num_classes=2, dropout=dropout).to(device)
    else:
        raise ValueError("model_name must be 'gcn' or 'transformer'")

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val = -math.inf
    best_state = None

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device, cw)
        val_metrics = evaluate(model, val_loader, device)
        val_score = val_metrics["auroc"] if not math.isnan(val_metrics["auroc"]) else val_metrics["acc"]

        if val_score > best_val:
            best_val = val_score
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0 or epoch == 1:
            print(f"[{epoch:03d}] loss={train_loss:.4f}  val_acc={val_metrics['acc']:.4f}  val_auroc={val_metrics['auroc']:.4f}")

    if best_state is not None:
        # save best state to disk so the artifact persists after the run
        save_path = os.path.join(os.getcwd(), "gnns_best_state.pt")
        torch.save(best_state, save_path)
        print(f"Saved best state to {save_path}")
        model.load_state_dict(best_state)

    test_metrics = evaluate(model, test_loader, device)
    print(f"TEST  acc={test_metrics['acc']:.4f}  auroc={test_metrics['auroc']:.4f}")
    return test_metrics


if __name__ == "__main__":
    # >>>> EDIT THIS to your graphs folder <<<<
    root_dir = "/dsi/efroni-lab/sbm/chen2/graph_data/10000"

    main(
        root_dir=root_dir,
        # model_name="gcn",
        model_name="transformer",
        hidden=64,
        dropout=0.2,
        batch_size=16,
        max_epochs=100,
        lr=1e-3,
        weight_decay=5e-4,
        val_ratio=0.2,
        test_ratio=0.2,
        seed=42,
    )


Using device: cuda
[001] loss=0.6904  val_acc=0.5556  val_auroc=0.4750
[001] loss=0.6904  val_acc=0.5556  val_auroc=0.4750
[010] loss=0.6890  val_acc=0.5000  val_auroc=0.7125
[010] loss=0.6890  val_acc=0.5000  val_auroc=0.7125
[020] loss=0.6882  val_acc=0.5000  val_auroc=0.7250
[020] loss=0.6882  val_acc=0.5000  val_auroc=0.7250
[030] loss=0.6971  val_acc=0.6111  val_auroc=0.7375
[030] loss=0.6971  val_acc=0.6111  val_auroc=0.7375
[040] loss=0.6859  val_acc=0.7222  val_auroc=0.7250
[040] loss=0.6859  val_acc=0.7222  val_auroc=0.7250
[050] loss=0.6806  val_acc=0.6667  val_auroc=0.7125
[050] loss=0.6806  val_acc=0.6667  val_auroc=0.7125
[060] loss=0.6891  val_acc=0.6667  val_auroc=0.7000
[060] loss=0.6891  val_acc=0.6667  val_auroc=0.7000
[070] loss=0.6819  val_acc=0.5556  val_auroc=0.7000
[070] loss=0.6819  val_acc=0.5556  val_auroc=0.7000
[080] loss=0.6747  val_acc=0.6111  val_auroc=0.7000
[080] loss=0.6747  val_acc=0.6111  val_auroc=0.7000
[090] loss=0.6723  val_acc=0.7778  val_auroc=